# Mini-Project 2: Uganda Tax Engine

An object-oriented, Decimal-only tax calculator. Reusable domain logic is implemented in `src/tax_engine.py`; this notebook configures the examples, explains decisions, and presents calculations. Run it from the repository root with **Restart & Run All**.

**Rate note:** The resident PAYE bands supplied in the assignment conflict with the published FY2025/26 resident schedule referenced below. `ASSIGNMENT-2025/26` is implemented as an explicit instructional profile to satisfy the requested band-edge calculations; it is not represented as the official URA schedule. `URA-2025/26` is also available for resident and non-resident calculations. Rates and terms can change: verify the current law and URA guidance before real payroll or business use.

In [8]:
from datetime import date
from decimal import Decimal

TAX_DATE = date(2025, 10, 1)  # Inside the configured FY2025/26 schedules.
SCHEDULE_VERSION = 'ASSIGNMENT-2025/26'
MONTHLY_GROSS = Decimal('1200000')
MONTHLY_ALLOWANCES = Decimal('100000')
MONTHLY_LOCAL_SERVICE_TAX = Decimal('5000')  # Illustrative/configurable input.

In [9]:
from src.tax_engine import (
    CorporateIncomeTax,
    FlatWithholdingTax,
    Invoice,
    InvoiceDirection,
    LocalServiceTax,
    Money,
    PAYETax,
    PayrollCalculator,
    Residency,
    SupplyType,
    TaxContext,
    TaxPlanner,
    VAT,
)

def ugx(value: Decimal) -> Money:
    """Create a UGX Money value from a Decimal amount."""
    return Money(value)

print('Tax client loaded. All monetary inputs below are Decimal-backed Money values.')

Tax client loaded. All monetary inputs below are Decimal-backed Money values.


## Money representation and float precision

The `Money` value object rejects floats and rounds to whole UGX with `ROUND_HALF_UP`. Decimal arithmetic is constructed from decimal strings, not binary float literals. This avoids using an inexact binary representation for payment amounts.

In [10]:
float_sum = 0.1 + 0.2
decimal_sum = Decimal('0.1') + Decimal('0.2')
print('Binary float:', repr(float_sum), 'equal to 0.3?', float_sum == 0.3)
print('Decimal:', decimal_sum, 'equal to Decimal("0.3")?', decimal_sum == Decimal('0.3'))
print('Money rounding:', Money(Decimal('1.5')), 'and', Money(Decimal('1.4')))
assert float_sum != 0.3
assert decimal_sum == Decimal('0.3')
assert Money(Decimal('1.5')) == Money(Decimal('2'))

Binary float: 0.30000000000000004 equal to 0.3? False
Decimal: 0.3 equal to Decimal("0.3")? True
Money rounding: UGX 2 and UGX 1


## PAYE: assignment bands, boundary tests, and residence

The table below evaluates the exact requested resident assignment schedule around its six threshold values. The code carries separate resident and non-resident bands. The `URA-2025/26` profile uses monthly equivalents of the published FY2025/26 annual schedules; compare it with the assignment profile rather than confusing the two.

In [11]:
assignment_edges = [
    335_000, 335_001, 410_000, 485_000, 10_000_000, 10_000_001,
]
assignment_paye = PAYETax()
for monthly_income in assignment_edges:
    result = assignment_paye.calculate(
        TaxContext(
            income=ugx(Decimal(monthly_income)),
            tax_date=TAX_DATE,
            residency=Residency.RESIDENT,
            schedule_version=SCHEDULE_VERSION,
        )
    )
    print(f'UGX {monthly_income:>10,} -> PAYE {result.amount}')

statutory_profile = PAYETax().calculate(
    TaxContext(
        income=ugx(Decimal('335000')),
        tax_date=TAX_DATE,
        residency=Residency.RESIDENT,
        schedule_version='URA-2025/26',
    )
)
nonresident_profile = PAYETax().calculate(
    TaxContext(
        income=ugx(Decimal('335000')),
        tax_date=TAX_DATE,
        residency=Residency.NON_RESIDENT,
        schedule_version='URA-2025/26',
    )
)
print('Published FY2025/26 reference profile at UGX 335,000:')
print('  Resident:', statutory_profile.amount)
print('  Non-resident:', nonresident_profile.amount)

UGX    335,000 -> PAYE UGX 0
UGX    335,001 -> PAYE UGX 0
UGX    410,000 -> PAYE UGX 15,000
UGX    485,000 -> PAYE UGX 33,750
UGX 10,000,000 -> PAYE UGX 2,888,250
UGX 10,000,001 -> PAYE UGX 2,888,250
Published FY2025/26 reference profile at UGX 335,000:
  Resident: UGX 10,000
  Non-resident: UGX 33,500


## Payroll pipeline

The pipeline computes configured Local Service Tax (LST) first and subtracts it from employment income before PAYE, as required by this project brief. Employee NSSF is 5% of gross pay and is deducted from take-home pay after the PAYE base is established; the employer's 10% contribution is reported separately and is not deducted from the employee. This order is an explicit model assumption for the assignment and must be checked against current law and payroll guidance before real use.

In [12]:
payroll = PayrollCalculator(
    paye=PAYETax(),
    local_service_tax=LocalServiceTax(ugx(MONTHLY_LOCAL_SERVICE_TAX)),
)
employee_context = TaxContext(
    income=ugx(MONTHLY_GROSS),
    allowances=ugx(MONTHLY_ALLOWANCES),
    tax_date=TAX_DATE,
    residency=Residency.RESIDENT,
    schedule_version=SCHEDULE_VERSION,
)
payroll_result = payroll.calculate(employee_context)
for label, amount in (
    ('Gross pay including allowances', payroll_result.gross_pay),
    ('Employee NSSF (5%)', payroll_result.nssf_employee),
    ('Employer NSSF (10%, not deducted)', payroll_result.nssf_employer),
    ('Configured LST', payroll_result.local_service_tax),
    ('PAYE after LST', payroll_result.paye),
    ('Net pay', payroll_result.net_pay),
):
    print(f'{label}: {amount}')

Gross pay including allowances: UGX 1,300,000
Employee NSSF (5%): UGX 65,000
Employer NSSF (10%, not deducted): UGX 130,000
Configured LST: UGX 5,000
PAYE after LST: UGX 276,750
Net pay: UGX 953,250


## VAT: standard-rated, zero-rated, and exempt invoices

Output VAT is calculated only for standard-rated sales at 18%. Zero-rated and exempt sales have no output VAT. The example credits input VAT only on standard-rated purchases. This small model assumes full recoverability on eligible purchases; real mixed taxable/exempt activities can require input-tax apportionment and supporting documentation.

In [13]:
invoices = (
    Invoice(ugx(Decimal('1000000')), SupplyType.STANDARD_RATED, InvoiceDirection.SALE),
    Invoice(ugx(Decimal('250000')), SupplyType.ZERO_RATED, InvoiceDirection.SALE),
    Invoice(ugx(Decimal('150000')), SupplyType.EXEMPT, InvoiceDirection.SALE),
    Invoice(
        ugx(Decimal('300000')), SupplyType.STANDARD_RATED,
        InvoiceDirection.PURCHASE, input_vat=ugx(Decimal('54000')),
    ),
)
vat_result = VAT().calculate(TaxContext(income=ugx(Decimal('0')), invoices=invoices))
print('Output VAT:', vat_result.breakdown['output_vat'])
print('Recoverable input VAT:', vat_result.breakdown['recoverable_input_vat'])
print('Net VAT payable:', vat_result.amount)

Output VAT: UGX 180,000
Recoverable input VAT: UGX 54,000
Net VAT payable: UGX 126,000


## Corporate income tax and extension: TaxPlanner

Corporate income tax uses revenue less allowable deductions and a 30% illustrative standard rate. This simplified class does not implement presumptive taxation or special company regimes. The chosen extension is **B: TaxPlanner**, which compares a payroll salary scenario with a dividend scenario using an explicitly configured withholding rate. The example is a comparison tool, not personal financial advice; tax eligibility, company profits, NSSF, dividend declarations, and special rules are not modelled.

In [14]:
corporate_result = CorporateIncomeTax().calculate(
    TaxContext(
        income=ugx(Decimal('10000000')),
        allowable_deductions=ugx(Decimal('4000000')),
        tax_date=TAX_DATE,
    )
)
print('Corporate chargeable income:', corporate_result.breakdown['chargeable_income'])
print('Corporate income tax (30%):', corporate_result.amount)

planner = TaxPlanner(payroll=payroll)
salary_scenario = planner.salary_scenario('Salary', employee_context)
dividend_scenario = planner.dividend_scenario(
    'Dividend',
    ugx(Decimal('1300000')),
    TAX_DATE,
    FlatWithholdingTax(Decimal('0.15'), name='Dividend withholding'),
)
comparison = planner.compare(salary_scenario, dividend_scenario)
print(f'Salary net: {salary_scenario.net}; dividend net: {dividend_scenario.net}')
print(
    f"Higher net in this simplified example: {comparison.preferred_scenario}; "
    f"difference: {comparison.net_difference}"
)

Corporate chargeable income: UGX 6,000,000
Corporate income tax (30%): UGX 1,800,000
Salary net: UGX 953,250; dividend net: UGX 1,105,000
Higher net in this simplified example: Dividend; difference: UGX 151,750


## Tests and reproducibility

Run the test suite from the project root with `python -m pytest tests -q`. The tests check every required resident assignment band edge, Decimal-only money operations, PAYE schedule dates, both residency schedules, payroll deduction order, VAT classifications, corporate tax, and the scenario comparison. Hypothesis generates many income values to check that PAYE is non-negative, monotonic, and never exceeds income. The tests use no network calls or random-number generation; Hypothesis records failing examples for reproducibility.

## Findings and design discussion

### Scope and regulatory baseline

The engine is deliberately small and explicit. `TaxContext` carries the tax date, residency, incomes, deductions, and invoices. The abstract `Tax` contract returns a `TaxResult` with both an amount and calculation breakdown. PAYE uses a named, date-bounded schedule rather than hidden constants, and raises `NoScheduleForDateError` outside the configured period. The `Money` value object represents UGX with `Decimal`, rejects floats, and rounds to a whole shilling using `ROUND_HALF_UP`. Taxes therefore share a single rounding boundary. PAYE reports the selected schedule version and its source note, so a result can be audited.

The assignment's example resident schedule is implemented as `ASSIGNMENT-2025/26`, including its 10% top-band surcharge (a 40% marginal rate above UGX 10 million). It is not silently labelled official. A separate `URA-2025/26` profile converts published annual resident and non-resident band figures into monthly equivalents. The public FY2025/26 tax table lists resident annual thresholds UGX 2.82m, 4.02m, 4.92m, and 120m; the non-resident table begins at UGX 4.02m. This conflicts with the task's resident nil threshold of UGX 335,000 and its extra UGX 485,000 threshold. At UGX 335,000 the assignment profile returns nil resident PAYE, while the published-table profile returns UGX 10,000 for a resident and UGX 33,500 for a non-resident. That concrete difference is why two profiles are named and why the assignment example does not claim legal authority. The FY2025/26 annual figures were checked against PwC's Uganda tax summary on 12 October 2026 and should be reconciled with URA's latest official guidance and legislation before actual use. Primary-law reference: Uganda Income Tax Act 1997, Cap. 340, Third Schedule, as amended; [URA website](https://ura.go.ug/). Comparative published schedule: [PwC Uganda — taxes on personal income](https://taxsummaries.pwc.com/uganda/individual/taxes-on-personal-income).

The payroll pipeline follows the assignment's requested order: configurable LST is applied before PAYE, employee NSSF (5%) is then deducted in computing take-home pay, and employer NSSF (10%) is reported separately. This distinction prevents employer contributions being accidentally removed from an employee's pay. LST is an input, not a hard-coded legal tariff. Payroll rules depend on current law and worker circumstances, so this sequence and rates are an educational model, not an official payroll determination. The notebook uses decimal strings for all amounts.

The VAT model applies the standard 18% rate to standard-rated sales, no output VAT to zero-rated or exempt sales, and credits input VAT only for standard-rated purchases. Net VAT may be negative when credits exceed output VAT. The example assumes full input-credit recovery; actual mixed supplies may require apportionment, eligibility checks, e-invoices, and filing controls. The 30% corporate module computes revenue less allowable deductions and floors chargeable income at zero. It omits presumptive regimes and other company-specific adjustments. The chosen extension is TaxPlanner: it compares a payroll scenario to a dividend scenario with a configured 15% withholding assumption. The rate is an assumption for the illustration, not a determination that a particular dividend is taxable at that rate. See [PwC Uganda — corporate taxes](https://taxsummaries.pwc.com/uganda/corporate/taxes-on-corporate-income) and [VAT](https://taxsummaries.pwc.com/uganda/corporate/other-taxes); consult the [URA](https://ura.go.ug/) and relevant current acts for authoritative rates, exemptions, filing rules, and effective dates.

### Why float is unsafe for tax

Binary floating point stores many decimal fractions approximately. The notebook demonstrates that Python evaluates `0.1 + 0.2` as a value whose representation is `0.30000000000000004`, and the comparison with `0.3` is false. Repeatedly multiplying, subtracting, and rounding these approximations at progressive thresholds can create a one-shilling discrepancy or move an amount on a boundary. `Decimal('0.1') + Decimal('0.2')` is exactly `Decimal('0.3')`. The `Money` class accepts only `Decimal`, rounds once per produced UGX amount with a stated half-up rule, and implements addition, multiplication, comparison, and a clear representation. Inputs should be made from strings (for example `Decimal('335001')`), never from `Decimal(335001.0)`.

### OOP choices

Inheritance is justified at the tax-calculation boundary: PAYE, NSSF, Local Service Tax, VAT, corporate income tax, and flat withholding each implement the same abstract `Tax.calculate(ctx)` contract while keeping their distinct calculations isolated. `PayrollCalculator` uses composition because payroll is an ordered pipeline of several independent taxes, not a special kind of tax. `TaxPlanner` also composes the payroll and withholding components. `Money` is a value object with immutable state and arithmetic dunder methods because an amount should not change after it has been validated; `TaxContext`, invoices, schedules, and results are explicit typed records. The design avoids making a deep class hierarchy for individual rates or supplies: rates and supply treatments are data, selected through enums and versioned schedules.

### Reliability and limitations

Invalid or negative monetary inputs are surfaced through `InvalidIncomeError`; date ranges without a schedule raise `NoScheduleForDateError`; both inherit from `TaxError`. This is preferable to returning an apparently successful zero for an unsupported year. Tests combine exact boundaries with Hypothesis properties: tax remains non-negative, does not decrease as income rises, and never exceeds gross income for the configured schedules. Fixed seeds are unnecessary because the notebook and tests do not generate random data; Hypothesis manages and reports its own examples. The project is still not a substitute for current URA publications or professional review: the displayed law profile is date-bounded to FY2025/26, invoices and input VAT are simplified, and deductions, reliefs, sector exceptions, turnover thresholds, and taxpayer-specific facts are outside scope.